# probe_whisper_attention_padding — probe diagnóstico do item 11 (ARCH-04 residual)

Pré-registro em `improvements/19_WHISPER_PADDING_PROBE_SPEC.md` — LER ANTES desta célula 0
mudar de sentido. Este notebook **não corrige nada** e **não grava nenhum `.pt` novo em
`audio_features/`**: é diagnóstico, decide se a self-attention do encoder do Whisper dentro
da janela de 30s (que a correção A1/A2 de `extract_audio_v2.ipynb` não toca — A1/A2 corrigem
a MÉDIA, não a atenção) é o mecanismo do *nuisance* de duração medido no doc 16
(AUC−nulo = 0,985 para decodificar duração a partir do áudio v2).

Dois caminhos, mesma amostra (~300–400 itens `training` com `duration_s < 30`):
- **Caminho A** — cópia declarada de `extract_whisper_features_v2` (produção, v2 em disco),
  com portão de cosseno > 0,999 contra o `.pt` salvo.
- **Caminho B** — mesmo mel, cortado para `2·n_valid` colunas ANTES do encoder; forward
  manual do `AudioEncoder` com `positional_embedding` fatiado — o padding de silêncio nunca
  entra em nenhum bloco de atenção.

Leitura pré-registada (3 condições, doc 19 §3) — não decidida aqui, só reportada.

In [ ]:
# P1-fix: os caminhos deste notebook são RELATIVOS a EXIST_VIDEOS/ (plano 11 §3).
# Se o kernel nascer noutro cwd, os asserts abaixo falham. Idempotente.
# Padrão copiado de probe_video_contribution.ipynb célula 0 (mesma convenção de projeto).
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretório do projeto não encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
assert Path("video_core.py").is_file(), "video_core.py não está no cwd"
print("cwd:", Path.cwd())

# ── Log em arquivo (independente de o Jupyter gravar outputs ou não) ────────
# O stdout é DUPLICADO: continua aparecendo na célula E vai para o arquivo.
# `_ORIG_STDOUT` guardado em globals() para re-executar esta célula não empilhar Tees.
import sys

_ORIG_STDOUT = globals().get("_ORIG_STDOUT", sys.stdout)


class _Tee:
    def __init__(self, path, term):
        self.f = open(path, "w", buffering=1, encoding="utf-8")
        self.term = term

    def write(self, s):
        self.term.write(s)
        self.f.write(s)
        return len(s)

    def flush(self):
        self.term.flush()
        self.f.flush()

    def isatty(self):
        return False


Path("analysis").mkdir(exist_ok=True)
LOG_PATH = "analysis/whisper_padding_probe_report.txt"
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
print("[log] stdout duplicado para", LOG_PATH,
      "\n[log] stderr NAO e capturado — se houver traceback, copiar da tela")


## 1. Imports, GPU, carregamento do Whisper large-v3

In [ ]:
import json as _json
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import whisper

import video_core as vc  # só para vc.split_val_halves — nenhum modelo de vídeo é carregado

warnings.filterwarnings("ignore", category=UserWarning)

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", ("Este probe exige GPU (encoder do Whisper large-v3). Roda na VM, "
                          "não na máquina local de desenvolvimento.")

WHISPER_MODEL = "large-v3"      # idêntico a extract_audio_v2.ipynb — NÃO MUDAR
TARGET_SR     = 16000
CHUNK_SECONDS = 30
CHUNK_SAMPLES = CHUNK_SECONDS * TARGET_SR      # 480 000
N_ENC_FRAMES  = 1500
FRAMES_PER_SEC = N_ENC_FRAMES // CHUNK_SECONDS  # 50

print(f"Carregando Whisper {WHISPER_MODEL} em {device}...")
model = whisper.load_model(WHISPER_MODEL, device=device)
model.eval()
_enc_dtype = next(model.encoder.parameters()).dtype
print(f"dtype dos parâmetros do encoder: {_enc_dtype}  (o forward manual do caminho B "
      f"casa dtype explicitamente contra isto, não assume fp16 nem fp32)")


## 2. Amostra — itens `training` com `duration_s < 30`

Filtra `analysis/audio_durations.csv` (`split == "training"`, `duration_s < 30`), amostra
~300–400 IDs com seed fixa, e checa que os `.pt` do v2 já extraídos existem em
`audio_features/v2/training/` para todos eles (pré-condição do portão do Caminho A —
§4.1 do doc 19).

In [ ]:
SAMPLE_SEED = 20260813
N_SAMPLE = 350

dur = pd.read_csv("analysis/audio_durations.csv")
dur["id_EXIST"] = dur["id_EXIST"].astype(str)
pool = dur[(dur["split"] == "training") & (dur["duration_s"] < 30) & (dur["status"] == "ok")].copy()
print(f"pool: {len(pool)} itens training com duration_s < 30 (esperado ~1682, doc 19 §5)")

rng = np.random.RandomState(SAMPLE_SEED)
n_sample = min(N_SAMPLE, len(pool))
sample_idx = rng.choice(len(pool), size=n_sample, replace=False)
sample = pool.iloc[sample_idx].reset_index(drop=True)
print(f"amostra: {len(sample)} itens (seed {SAMPLE_SEED})")

AUDIO_V2_DIR = Path("audio_features/v2/training")
VIDEO_DIR = Path("Dataset/training/videos")
missing_pt = [i for i in sample["id_EXIST"] if not (AUDIO_V2_DIR / f"{i}.pt").is_file()]
assert not missing_pt, (
    f"{len(missing_pt)} itens da amostra sem .pt em {AUDIO_V2_DIR} — pré-condição do "
    f"portão do Caminho A falhou antes de começar. IDs: {missing_pt[:20]}")
print(f"todos os {len(sample)} .pt do v2 confirmados em {AUDIO_V2_DIR}")

# metadados (lang, A_hard) vêm do JSON de treino — mesma fonte que build_frozen_split usa.
full_df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
full_df["id_EXIST"] = full_df["id_EXIST"].astype(str)
full_df = full_df.set_index("id_EXIST", drop=False)

sample_ids = list(sample["id_EXIST"])
missing_meta = [i for i in sample_ids if i not in full_df.index]
assert not missing_meta, f"IDs da amostra ausentes do JSON de treino: {missing_meta[:20]}"

sample_lang = full_df.loc[sample_ids, "lang"].values
sample_a_hard = np.array([
    1 if "YES" in list(full_df.loc[i, "labels_task3_1"]) else 0
    for i in sample_ids
])  # mesma regra de vc._task_a_hard (usada só p/ estratificar o split, não p/ treino)
sample_dur = sample["duration_s"].values

print("\ndistribuição da amostra:")
print(f"  língua: {dict(zip(*np.unique(sample_lang, return_counts=True)))}")
print(f"  A_hard: {dict(zip(*np.unique(sample_a_hard, return_counts=True)))}")
print(f"  duração: mediana {np.median(sample_dur):.2f}s, min {sample_dur.min():.2f}s, "
      f"max {sample_dur.max():.2f}s")


## 3. Caminho A — cópia declarada de `extract_whisper_features_v2`

**Cópia intencional** das ~20 linhas da célula 3 de `extract_audio_v2.ipynb` (o notebook de
produção real, só leitura — não editado, não importado, porque não é um módulo). Qualquer
divergência de comportamento entre este bloco e o notebook de produção invalidaria o portão
de cosseno abaixo, então o texto é reproduzido literalmente, sem "melhorias".

Para os itens desta amostra (`duration_s < 30`) o `for start in range(...)` do original roda
exatamente 1 vez — a agregação ponderada entre janelas (A2) é mantida por fidelidade à cópia,
mas é uma no-op aqui.

In [ ]:
resampler_cache = {}


def load_audio_mono_16k(video_path: Path):
    """Cópia declarada de extract_audio_v2.ipynb célula 3 (load_audio_mono_16k).
    Waveform [T] float32 mono a 16 kHz, ou None em falha."""
    try:
        waveform, sr = torchaudio.load(str(video_path))        # [C, T]
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)      # [1, T]
        if sr != TARGET_SR:
            if sr not in resampler_cache:
                resampler_cache[sr] = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform = resampler_cache[sr](waveform)
        return waveform.squeeze(0)                             # [T]
    except Exception as e:
        print(f"  [ERR] torchaudio.load falhou: {e}")
        return None


@torch.no_grad()
def extract_whisper_features_v2(waveform: torch.Tensor):
    """CÓPIA DECLARADA de extract_audio_v2.ipynb célula 3 — não é importação, é
    reprodução literal (comentário obrigatório do plano: notebook de produção não pode
    ser editado nem importado como módulo). Qualquer mudança de comportamento aqui
    invalidaria o portão de cosseno da célula seguinte.

    A1 — dentro de cada janela de 30 s, a média usa só os primeiros
         n_valid = round(segundos_válidos × 50) frames do encoder.
    A2 — agregação entre janelas ponderada por n_valid (no-op para duration_s < 30,
         que é a única condição usada nesta amostra — sempre 1 janela).

    Devolve (features [1280] float32 em CPU, n_windows) ou (None, 0) se vazio.
    """
    n_samples = int(waveform.shape[0])
    if n_samples == 0:
        return None, 0
    feats, weights = [], []
    for start in range(0, n_samples, CHUNK_SAMPLES):
        chunk = waveform[start:start + CHUNK_SAMPLES]
        valid_seconds = chunk.shape[0] / TARGET_SR
        n_valid = max(1, min(N_ENC_FRAMES, int(round(valid_seconds * FRAMES_PER_SEC))))
        audio = whisper.pad_or_trim(chunk.numpy())             # [480000]
        mel = whisper.log_mel_spectrogram(
            audio, n_mels=model.dims.n_mels).to(device)        # [n_mels, 3000]
        enc = model.encoder(mel.unsqueeze(0))                  # [1, 1500, 1280]
        feats.append(enc[0, :n_valid].mean(dim=0))             # A1: só frames válidos
        weights.append(float(n_valid))
    w = torch.tensor(weights, device=feats[0].device)
    features = (torch.stack(feats, dim=0) * w[:, None]).sum(dim=0) / w.sum()  # A2
    return features.cpu().float(), len(feats)


print("Caminho A definido (cópia declarada de extract_whisper_features_v2).")


## 4. Caminho B — mel cortado para `2·n_valid` colunas, forward manual do encoder

A API de alto nível `model.encoder(mel)` exige `mel.shape[-1] == 1500` (o
`positional_embedding` é uma tabela fixa somada por broadcasting, com um `assert` interno de
shape na classe `AudioEncoder`). Para processar só os `n_valid` frames válidos — sem o
padding de silêncio nunca entrar em nenhum bloco de atenção — o forward do encoder é
replicado manualmente: `conv1 → gelu → conv2 → gelu → permute → +
positional_embedding[:n_valid] → blocks → ln_post`.

`conv1` é `stride=1, padding=1` (preserva o comprimento); `conv2` é `stride=2, padding=1`
(divide por 2). Cortar o mel para `2·n_valid` colunas produz exatamente `n_valid` posições na
saída — casa exatamente com `positional_embedding[:n_valid]`, sem interpolação.

`[VERIFICAR NA VM]` — os nomes de atributo abaixo (`model.encoder.conv1`, `.conv2`,
`.positional_embedding`, `.blocks`, `.ln_post`) são os da classe `AudioEncoder` de
`whisper/model.py` no pacote `openai-whisper` tal como documentada/lida por esta sessão (sem
o pacote instalado localmente para confirmar em runtime — CLAUDE.md regista
`openai-whisper==20250625` na VM, colhido em 2026-08-09, mas o nome exato dos atributos
internos não foi reconferido nesta sessão). Se `AttributeError`, a célula abaixo já imprime
`dir(model.encoder)` antes de abortar — ajustar os nomes ali, não adivinhar de novo.

In [ ]:
def _check_encoder_attrs(encoder):
    required = ["conv1", "conv2", "positional_embedding", "blocks", "ln_post"]
    missing = [a for a in required if not hasattr(encoder, a)]
    if missing:
        print("ATRIBUTOS FALTANDO no AudioEncoder:", missing)
        print("dir(model.encoder):", [a for a in dir(encoder) if not a.startswith("_")])
        raise AttributeError(
            f"[VERIFICAR NA VM] nomes de atributo divergem do esperado: {missing}. "
            "Ajustar _whisper_encoder_forward_manual com os nomes reais acima.")
    print("atributos do AudioEncoder confirmados:", required)


_check_encoder_attrs(model.encoder)


@torch.no_grad()
def _whisper_encoder_forward_manual(encoder, mel_slice: torch.Tensor, n_valid: int):
    """Forward manual do AudioEncoder (whisper.model.AudioEncoder), com
    positional_embedding FATIADO em vez do comprimento fixo n_ctx=1500 que a API de
    alto nível exige. [VERIFICAR NA VM]: ver célula anterior — nomes de atributo
    confirmados por _check_encoder_attrs antes de esta função rodar.

    mel_slice: [n_mels, 2*n_valid] (já cortado — NÃO passar o mel de 3000 colunas aqui).
    Devolve [1, n_valid, n_state].
    """
    dtype = next(encoder.parameters()).dtype
    x = mel_slice.unsqueeze(0).to(dtype)                    # [1, n_mels, 2*n_valid]
    x = F.gelu(encoder.conv1(x))
    x = F.gelu(encoder.conv2(x))
    x = x.permute(0, 2, 1)                                  # [1, L_out, n_state]
    assert x.shape[1] == n_valid, (
        f"corte de mel não produziu n_valid posições: {x.shape[1]} != {n_valid}. "
        "Conferir a aritmética de conv1(stride1,pad1)+conv2(stride2,pad1) — doc 19 §4.2.")
    pos = encoder.positional_embedding[:n_valid].to(dtype)
    x = (x + pos).to(x.dtype)
    for block in encoder.blocks:
        x = block(x)
    x = encoder.ln_post(x)
    return x                                                # [1, n_valid, n_state]


@torch.no_grad()
def extract_whisper_features_pathB(waveform: torch.Tensor):
    """Caminho B: mesmo mel do Caminho A, cortado para 2*n_valid colunas ANTES do
    encoder. Só janela única — restrito por construção a duration_s < 30 (§5 do doc 19),
    então não há agregação entre janelas a replicar aqui."""
    n_samples = int(waveform.shape[0])
    if n_samples == 0:
        return None
    assert n_samples <= CHUNK_SAMPLES, (
        "Caminho B assume janela única (duration_s < 30); amostra violou a restrição.")
    valid_seconds = n_samples / TARGET_SR
    n_valid = max(1, min(N_ENC_FRAMES, int(round(valid_seconds * FRAMES_PER_SEC))))

    audio = whisper.pad_or_trim(waveform.numpy())               # [480000] (só p/ mel completo)
    mel = whisper.log_mel_spectrogram(
        audio, n_mels=model.dims.n_mels).to(device)             # [n_mels, 3000]
    mel_cut = mel[:, : 2 * n_valid]                              # corte ANTES do encoder

    enc = _whisper_encoder_forward_manual(model.encoder, mel_cut, n_valid)  # [1, n_valid, 1280]
    features = enc[0].mean(dim=0)                                # média simples: já são só válidos
    return features.cpu().float()


print("Caminho B definido (forward manual, positional_embedding fatiado).")


## 5. Rodar os dois caminhos sobre a amostra (cache idempotente)

Portão do Caminho A (obrigatório, aborta se falhar): cosseno > 0,999 contra o `.pt` já salvo
em `audio_features/v2/training/{id}.pt` para os mesmos IDs — se esse portão falhar, os dois
caminhos ficam sem valor probatório (investigar dtype/versão do Whisper antes de ler
qualquer AUC, doc 19 §4.1).

In [ ]:
CACHE_FILE = Path("analysis/whisper_padding_probe_cache.pt")

if CACHE_FILE.exists():
    cache = torch.load(CACHE_FILE, weights_only=False)
    print(f"[cache] carregado de {CACHE_FILE} (extração saltada)")
    assert cache["sample_ids"] == sample_ids, (
        "cache antigo com amostra diferente — apagar analysis/whisper_padding_probe_cache.pt "
        "e re-rodar")
else:
    feats_a, feats_b, cos_gate = [], [], []
    n_windows_hist = {}
    file_by_id = dict(zip(sample["id_EXIST"], sample["file"]))
    for item_id in sample_ids:
        # nome de arquivo já resolvido por ffprobe em audio_durations.csv — mesma fonte
        # usada para construir a amostra, não precisa reconsultar o JSON.
        video_path = VIDEO_DIR / file_by_id[item_id]
        assert video_path.is_file(), f"vídeo não encontrado para {item_id}: {video_path}"

        waveform = load_audio_mono_16k(video_path)
        assert waveform is not None, f"torchaudio.load falhou para {item_id}"

        feat_a, n_windows = extract_whisper_features_v2(waveform)
        assert feat_a is not None, f"Caminho A vazio para {item_id}"
        n_windows_hist[n_windows] = n_windows_hist.get(n_windows, 0) + 1

        feat_b = extract_whisper_features_pathB(waveform)

        # PORTÃO: cosseno do Caminho A recomputado contra o .pt de produção em disco.
        on_disk = torch.load(AUDIO_V2_DIR / f"{item_id}.pt", weights_only=False).float()
        cos = F.cosine_similarity(feat_a.unsqueeze(0), on_disk.unsqueeze(0)).item()
        cos_gate.append(cos)

        feats_a.append(feat_a)
        feats_b.append(feat_b)

    feats_a = torch.stack(feats_a)   # [N, 1280]
    feats_b = torch.stack(feats_b)   # [N, 1280]
    cos_gate = np.array(cos_gate)

    print(f"\nhistograma de janelas no Caminho A (esperado {{1: {len(sample_ids)}}}, "
          f"amostra restrita a duration_s < 30): {n_windows_hist}")
    print(f"PORTÃO cosseno A vs disco: min={cos_gate.min():.6f} "
          f"mediana={np.median(cos_gate):.6f} n<0.999={int((cos_gate < 0.999).sum())}")
    assert (cos_gate > 0.999).all(), (
        f"PORTÃO FALHOU: {int((cos_gate <= 0.999).sum())} itens com cosseno <= 0.999 contra "
        "o .pt em disco. NÃO seguir para as AUCs — investigar dtype/versão do Whisper "
        "primeiro (doc 19 §4.1). Abortando.")
    print("PORTÃO OK: Caminho A recomputado bate com o .pt de produção (cosseno > 0,999).")

    cache = {
        "sample_ids": sample_ids,
        "feats_a": feats_a, "feats_b": feats_b,
        "cos_gate": cos_gate,
        "sample_lang": sample_lang, "sample_a_hard": sample_a_hard, "sample_dur": sample_dur,
        "date": datetime.now(timezone.utc).isoformat(),
    }
    torch.save(cache, CACHE_FILE)
    print(f"[cache] gravado em {CACHE_FILE}")

FEATS_A = cache["feats_a"]
FEATS_B = cache["feats_b"]
assert FEATS_A.shape == FEATS_B.shape == (len(sample_ids), 1280)
print("shapes OK:", tuple(FEATS_A.shape), tuple(FEATS_B.shape))


## 6. Sonda linear (receita verbatim — célula 23 de `probe_video_contribution.ipynb`)

`StandardScaler` + `LogisticRegression(C=0.05)`, ajuste na metade `fit` de
`vc.split_val_halves(N)`, AUC relatado na metade `report`, nulo por 10 permutações do rótulo.
Roda para os dois caminhos (A, B) × três alvos (duração, língua, A_hard).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

N_SAMPLE_FINAL = len(sample_ids)
IDX_FIT, IDX_REP = vc.split_val_halves(N_SAMPLE_FINAL)
print(f"metades (vc.split_val_halves): fit={len(IDX_FIT)}  report={len(IDX_REP)}")

targets = {
    "duração > mediana": (cache["sample_dur"] > np.median(cache["sample_dur"])).astype(int),
    "língua (es vs en)": (cache["sample_lang"] == "es").astype(int),
    "A_hard (sexista?)": cache["sample_a_hard"].astype(int),
}
print("\ndistribuição dos alvos:")
for name, y in targets.items():
    u, c = np.unique(y, return_counts=True)
    print(f"  {name:22s} {dict(zip(u.tolist(), c.tolist()))}")


def probe(X, y, seed_shuffle=None):
    """AUC na metade de report; fit só na metade de fit. Cópia verbatim da função
    `probe()` de probe_video_contribution.ipynb célula 23 (2 classes só, aqui)."""
    if seed_shuffle is not None:
        y = np.random.RandomState(seed_shuffle).permutation(y)
    fit, rep = IDX_FIT, IDX_REP
    if len(np.unique(y[fit])) < 2 or len(np.unique(y[rep])) < 2:
        return float("nan")
    sc = StandardScaler().fit(X[fit])
    clf = LogisticRegression(max_iter=3000, C=0.05).fit(sc.transform(X[fit]), y[fit])
    p = clf.predict_proba(sc.transform(X[rep]))
    return float(roc_auc_score(y[rep], p[:, 1]))


paths = {"A (com padding na atenção)": FEATS_A.numpy(), "B (sem padding na atenção)": FEATS_B.numpy()}
probe_out = {}
print(f"\n{'alvo':>22s}" + "".join(f"{p:>34s}" for p in paths))
print(f"{'':>22s}" + "".join(f"{'AUC (AUC-nulo)':>34s}" for _ in paths))
for name, y in targets.items():
    row = f"{name:>22s}"
    for pname, X in paths.items():
        auc = probe(X, y)
        null = np.array([probe(X, y, seed_shuffle=1000 + r) for r in range(10)])
        gain = auc - np.nanmean(null)
        probe_out[(name, pname)] = (auc, float(np.nanmean(null)), gain)
        row += f"{auc:24.3f} ({gain:+.3f})"
    print(row)


## 7. Leitura pré-registada (doc 19 §3) + gravação do JSON final

In [ ]:
gain_dur_A = probe_out[("duração > mediana", "A (com padding na atenção)")][2]
gain_dur_B = probe_out[("duração > mediana", "B (sem padding na atenção)")][2]
gain_lang_A = probe_out[("língua (es vs en)", "A (com padding na atenção)")][2]
gain_lang_B = probe_out[("língua (es vs en)", "B (sem padding na atenção)")][2]
gain_ahard_A = probe_out[("A_hard (sexista?)", "A (com padding na atenção)")][2]
gain_ahard_B = probe_out[("A_hard (sexista?)", "B (sem padding na atenção)")][2]

delta_dur = gain_dur_B - gain_dur_A
delta_lang = gain_lang_B - gain_lang_A
delta_ahard = gain_ahard_B - gain_ahard_A

print("=" * 78)
print("RESUMO — probe de atenção ao padding do Whisper | n =", N_SAMPLE_FINAL,
      "| duration_s < 30 | training")
print("=" * 78)
print(f"\nΔ (AUC-nulo do B menos do A):")
print(f"  duração  (primário): {delta_dur:+.3f}   (A {gain_dur_A:+.3f} -> B {gain_dur_B:+.3f})")
print(f"  língua   (controle): {delta_lang:+.3f}   (A {gain_lang_A:+.3f} -> B {gain_lang_B:+.3f})")
print(f"  A_hard   (controle): {delta_ahard:+.3f}   (A {gain_ahard_A:+.3f} -> B {gain_ahard_B:+.3f})")

TOL = 0.05   # mesma tolerância usada na leitura de probe_video_contribution.ipynb célula 23
controls_stable = abs(delta_lang) <= TOL and abs(delta_ahard) <= TOL
duration_dropped_hard = delta_dur < -TOL

print("\nLEITURA (árvore pré-registada, doc 19 §3 — NÃO decidida antes desta célula rodar):")
if abs(delta_dur) <= TOL:
    print("  CONDIÇÃO 1: B decodifica duração IGUALMENTE BEM que A (|Δ| <= 0.05).")
    print("  -> ROTA FECHA. Achado negativo: a atenção ao padding não é o mecanismo do")
    print("     nuisance de duração — o Whisper já era robusto a ele (regime in-distribution")
    print("     de pré-treino, contra-argumento do doc 19 §2 confirmado). Nenhuma v3 é")
    print("     justificada por este eixo.")
elif duration_dropped_hard and controls_stable:
    print("  CONDIÇÃO 2: B piora MUITO em duração (Δ < -0.05) E os controles se MANTÊM")
    print("  (|Δ língua|, |Δ A_hard| <= 0.05).")
    print("  -> GATILHO, NUNCA PROVA, para considerar uma extração v3 que evite padding na")
    print("     atenção. Decisão de rodar v3 fica com o autor — este probe não a implementa.")
elif duration_dropped_hard and not controls_stable:
    print("  CONDIÇÃO 3: B piora em duração E TAMBÉM nos controles (língua e/ou A_hard caem")
    print("  junto).")
    print("  -> ROTA FECHA IGUAL — mas por degradação genérica do caminho B (regime")
    print("     out-of-distribution do encoder, contra-argumento do doc 19 §2), não por")
    print("     remoção seletiva de um confound. Não é evidência utilizável em nenhuma direção.")
else:
    print("  Nenhuma das 3 condições pré-registadas bate exatamente (ex.: duração melhora, ou")
    print("  padrão misto) — reportar os números brutos acima e NÃO forçar uma das 3 leituras.")

print("\n⚠ Sonda linear boa NÃO prova que o encoder usa essa informação em produção; sonda")
print("   ruim não prova que a informação não está lá em forma não-linear.")

out = {
    "n_sample": N_SAMPLE_FINAL,
    "sample_seed": SAMPLE_SEED,
    "cos_gate_min": float(cache["cos_gate"].min()),
    "cos_gate_median": float(np.median(cache["cos_gate"])),
    "probe_results": {
        f"{name} | {pname}": {"auc": v[0], "null_mean": v[1], "gain": v[2]}
        for (name, pname), v in probe_out.items()
    },
    "delta_B_minus_A": {
        "duração": delta_dur, "língua": delta_lang, "A_hard": delta_ahard,
    },
    "tolerance": TOL,
    "date": datetime.now(timezone.utc).isoformat(),
    "notebook": "probe_whisper_attention_padding.ipynb",
    "spec": "improvements/19_WHISPER_PADDING_PROBE_SPEC.md",
}
with open("analysis/whisper_padding_probe.json", "w") as f:
    _json.dump(out, f, indent=2, ensure_ascii=False)
print("\n[out] gravado em analysis/whisper_padding_probe.json")


## 8. Fechar o log (idempotente)

In [ ]:
# Restaura stdout original — idempotente: se _ORIG_STDOUT já é o stdout atual (célula 1
# nunca rodou nesta sessão, ou já foi fechado antes), não faz nada de mais.
if isinstance(sys.stdout, _Tee):
    _f = sys.stdout.f
    sys.stdout = _ORIG_STDOUT
    _f.close()
    print(f"[log] fechado — relatório completo em {LOG_PATH}")
else:
    print("[log] stdout já não é um _Tee (nada a fechar)")
